**Cell 1: Imports and Settings**

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report

# Settings
TRAIN_DIR = "train"
TEST_DIR  = "test"

IMG_SIZE = (128, 128)
BATCH_SIZE = 8
EPOCHS = 10


**CELL 2: Convert TIF → PNG**

In [ ]:
# def convert_tiff_to_png():
#     for split in ["train", "test"]:
#         for cls in ["folded"]:
#             folder = Path(split) / cls
#             for img_path in folder.glob("*.tif*"):
#                 img = cv2.imread(str(img_path))
#                 if img is None:
#                     continue
#                 png_path = img_path.with_suffix(".png")
#                 cv2.imwrite(str(png_path), img)
#                 os.remove(img_path)

# print("🔄 Converting TIFF images to PNG...")
# convert_tiff_to_png()
# print("✅ Conversion completed")

def convert_tif_to_png(folder):
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith((".tif", ".tiff")):
                tif_path = os.path.join(root, file)
                png_path = tif_path.replace(".tif", ".png").replace(".tiff", ".png")

                try:
                    img = Image.open(tif_path)
                    img.save(png_path)
                    print(f"Converted: {tif_path} -> {png_path}")
                except:
                    print(f"❌ ERROR converting: {tif_path}")

convert_tif_to_png("train")
convert_tif_to_png("test")


**CELL 3: Load Dataset**

In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    label_mode="binary"
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="binary"
)

class_names = train_ds.class_names
print("Classes:", class_names)


**CELL 4: CNN Model**

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(*IMG_SIZE, 3)),

    tf.keras.layers.Conv2D(16, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


**CELL 5: Train Model**

In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)


**CELL 6: Accuracy & Loss Graphs**

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label="Train Accuracy")
plt.plot(history.history['val_accuracy'], label="Test Accuracy")
plt.legend()
plt.title("Accuracy")

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label="Train Loss")
plt.plot(history.history['val_loss'], label="Test Loss")
plt.legend()
plt.title("Loss")

plt.show()


**CELL 7: Confusion Matrix**

In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)
    preds = (preds > 0.5).astype(int)
    y_pred.extend(preds.flatten())
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

#print(classification_report(y_true, y_pred, target_names=class_names))
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["clear", "folded"],
    zero_division=0
))

**CELL 8: Final Accuracy**

In [ ]:
train_acc = history.history['accuracy'][-1]
test_acc = history.history['val_accuracy'][-1]

print(f"Training Accuracy: {train_acc*100:.2f}%")
print(f"Testing Accuracy: {test_acc*100:.2f}%")


**CELL 9: Save Model**

In [ ]:
model.save("model_cnn.h5")
print("💾 Model saved as model1_cnn.h5")
